# Classification analysis (supermarkets data)

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn import tree
from sklearn.metrics import RocCurveDisplay
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

# Show version of scikit-learn
import sklearn
sklearn.__version__

## Import supermarkets data

In [ ]:
# Read supermarkets data
df_supermarkets = pd.read_csv("supermarkets_data_enriched.csv", sep=",", encoding="utf-8")

# Number of rows and columns
print(df_supermarkets.shape)

# First records
df_supermarkets.head(5)

## Variable description

- lat: latitude of supermarket location
- lon: longitude of supermarket location
- brand: supermarket brand name
- pop: population of the municipality
- pop_dens: population density
- frg_pct: percentage of foreign residents
- emp: employment in the municipality

## Count and remove missing values

In [ ]:
# Count missing values
print(df_supermarkets.isna().sum())

# Remove missing values
df_supermarkets = df_supermarkets.dropna(subset=['lat', 'lon', 'pop', 'pop_dens', 'frg_pct', 'emp', 'brand'])

# Number of rows after removing missing values
print(f'\nRows after removing missing values: {df_supermarkets.shape[0]}')

## Create subset with only Migros and Volg

In [ ]:
# Create a subset with only Migros and Volg brands
df_sub = df_supermarkets.loc[df_supermarkets['brand'].isin(['Migros', 'Volg'])]

# Show shape and value counts
print(f'Subset shape: {df_sub.shape}')
print(f'\nBrand value counts:')
print(df_sub['brand'].value_counts())

## Classification Tree
For details see: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

### Create train and test samples (train = 80%, test = 20% of the data)

In [ ]:
# Create train and test samples
X_train, X_test, y_train, y_test = train_test_split(df_sub[['lat', 
                                                            'lon',
                                                            'pop',
                                                            'pop_dens',
                                                            'frg_pct',
                                                            'emp']], 
                                                    df_sub['brand'], 
                                                    test_size=0.20, 
                                                    random_state=42)

# Show X_train
print('X_train:')
print(X_train.head(), '\n')

# Show y_train
print('y_train:')
print(y_train.head())

### Fit the classification tree model and make predictions

In [ ]:
# Initialize the classification tree model 
clf = DecisionTreeClassifier(random_state=20, 
                             max_depth=3)

# Train the classification tree model 
clf = clf.fit(X_train, y_train)

# Make model predictions
y_pred = clf.predict(X_test)
y_pred

### Print text representation of the classification tree

In [ ]:
# Text representation of the classification tree
text_rep = tree.export_text(clf, 
                            feature_names=list(X_train.columns))

# Print text_representation
print(text_rep)

### Visualize the classification tree

In [ ]:
# Visualize the classification tree
fig = plt.figure(figsize=(12,5))
tree_plot = tree.plot_tree(clf, 
                   feature_names=list(X_train.columns),  
                   class_names=['Migros', 'Volg'],
                   filled=True,
                   fontsize=10,
                   label='root')

### Show confusion matrix and classification report

In [ ]:
# Confusion matrix
print('Confusion matrix')
print(confusion_matrix(y_test, y_pred), '\n')

# Classification report
print('Classification report')
print(classification_report(y_test, y_pred))

### ROC curve and AUC

In [ ]:
# Encode labels for ROC curve (need numeric values)
le = LabelEncoder()
y_test_encoded = le.fit_transform(y_test)

# Plot ROC curve and calculate AUC for Classification Tree
plt.figure(figsize=(6,4))
ax = plt.gca()
ct_disp = RocCurveDisplay.from_estimator(clf, 
                                          X_test, 
                                          y_test_encoded, 
                                          ax=ax,
                                          alpha=0.8,
                                          c="darkblue")
plt.title('ROC Curve - Classification Tree')
plt.show()

## Random Forest Classifier
For details see: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

### Create train and test samples (train = 80%, test = 20% of the data)

In [ ]:
# Create train and test samples
X2_train, X2_test, y2_train, y2_test = train_test_split(df_sub[['lat', 
                                                                'lon',
                                                                'pop',
                                                                'pop_dens',
                                                                'frg_pct',
                                                                'emp']], 
                                                        df_sub['brand'], 
                                                        test_size=0.20, 
                                                        random_state=42)

# Show X2_train
print('X2_train:')
print(X2_train.head(), '\n')

# Show y2_train
print('y2_train:')
print(y2_train.head())

### Fit the Random Forest Classifier

In [ ]:
# Initialize the random forest classifier
rfc = RandomForestClassifier(random_state=20, max_depth=10)

# Train the random forest classifier
rfc = rfc.fit(X2_train, y2_train)

# Predict the target variable
y_pred_rf = rfc.predict(X2_test)

print('Predicted target variable (brand)')
y_pred_rf

### Show confusion matrix and classification report

In [ ]:
# Confusion matrix
print('Confusion matrix')
print(confusion_matrix(y2_test, y_pred_rf), '\n')

# Classification report
print('Classification report')
print(classification_report(y2_test, y_pred_rf))

### Show feature importance

In [ ]:
cols = X2_train.columns

# Derive feature importance from random forest
importances = rfc.feature_importances_
std = np.std([t.feature_importances_ for t in rfc.estimators_], axis=0)
indices = np.argsort(importances)[::-1]

# Print col-names and importances-values
print( cols[indices] )
print( importances[indices] )

# Barplot with feature importance
df_fi = pd.DataFrame({'features':cols,'importances': importances})
df_fi.sort_values('importances', inplace=True)
df_fi.plot(kind='barh', 
           y='importances', 
           x='features', 
           color='darkred', 
           figsize=(6,3))
plt.title('Feature Importance - Random Forest')
plt.show()

### ROC curve and AUC

In [ ]:
# Encode labels for ROC curve
le2 = LabelEncoder()
y2_test_encoded = le2.fit_transform(y2_test)

# Plot ROC curve and calculate AUC
plt.figure(figsize=(6,4))
ax = plt.gca()
rfc_disp = RocCurveDisplay.from_estimator(rfc, 
                                          X2_test, 
                                          y2_test_encoded, 
                                          ax=ax,
                                          alpha=0.8,
                                          c="darkred")
plt.title('ROC Curve - Random Forest')
plt.show()

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')